# 🔵 Environment Setup & Data Ingestion (Bronze Layer)


## Connect Google Drive in Colab (Mount Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load dataset using PySpark

In [ ]:
# Install Pyspark

! pip install pyspark

In [ ]:
# Create new Session - Pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("Bigdata Assignment 02") \
        .getOrCreate()

spark

## Load Dataset

In [53]:
# Read the CSV from Drive
csv_path = "/content/drive/MyDrive/Big Data/Assignment 02/Dataset.csv"
df = spark.read.csv(csv_path, header = True, inferSchema= True)

# Displaying the First 5 Rows
df.show(5)

# Display schema (Data Type)
df.printSchema()

# Display record counts
print("Total record count of dataset: ", df.count())



+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows
root

## Convert InvoiceDate into Timestamp

In [54]:
from pyspark.sql.functions import to_timestamp

bronze_df = df.withColumn(
    "InvoiceDate",
    to_timestamp("InvoiceDate", "M/d/yyyy HH:mm")
)

bronze_df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


## Partition Columns

In [ ]:
from pyspark.sql.functions import year, month

bronze_df = bronze_df \
    .withColumn("year", year("InvoiceDate")) \
    .withColumn("month", month("InvoiceDate"))

bronze_df.select("InvoiceDate","year","month").show(5)

+-------------------+----+-----+
|        InvoiceDate|year|month|
+-------------------+----+-----+
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
+-------------------+----+-----+
only showing top 5 rows



## Write Bronze Layer

In [ ]:
bronze_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/content/drive/MyDrive/Big Data/Assignment_2/bronze_parquet")

print("✅ Bronze Layer Created")

✅ Bronze Layer Created


# 🔵 Data Cleaning & Quality Management (Silver Layer)

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DecimalType

# # Convert Customer ID data type (double ---> integer) and UnitPrice (double ---> DecimalType(10,2))

silver_df = bronze_df \
    .withColumn("CustomerID", col("CustomerID").cast(IntegerType())) \
    .withColumn("UnitPrice", col("UnitPrice").cast(DecimalType(10,2))) # # double → floating point (can cause rounding errors)

silver_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



## Check Null values

In [ ]:
# New Null Values
from pyspark.sql.functions import sum as spark_sum

null_counts = silver_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

null_counts.show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+----+-----+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|year|month|
+---------+---------+-----------+--------+-----------+---------+----------+-------+----+-----+
|        0|        0|       1454|       0|          0|        0|    135080|      0|   0|    0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+----+-----+



## Missing CustomerID Handling

In [ ]:
from pyspark.sql.functions import col, sum, when, monotonically_increasing_id

total_records = silver_df.count()
print("Total records of the dataset:", total_records)

missing_customer_id = silver_df.filter(col("CustomerID").isNull())
missing_records = missing_customer_id.count()
print(f"Total records of missing CustomerID: {missing_records}")

Total records of the dataset: 541909
Total records of missing CustomerID: 135080


In [ ]:
from pyspark.sql.functions import col, when, row_number, monotonically_increasing_id
from pyspark.sql.window import Window

# Count missing CustomerID before processing
missing_customerID = silver_df.filter(col("CustomerID").isNull()).count()
print('Records with missing CustomerID (Before Processing):', missing_customerID)

# Get distinct invoices with missing CustomerID
missing_invoices = silver_df.filter(col("CustomerID").isNull()) \
                          .select("InvoiceNo") \
                          .distinct()

# Generate unique 5-digit negative IDs
window = Window.orderBy("InvoiceNo")
missing_invoices = missing_invoices.withColumn(
    "temp_id",
    row_number().over(window)
)
# Make them 5-digit negative numbers: -10001, -10002.
missing_invoices = missing_invoices.withColumn(
    "rep_customer_id",
    -(10000 + col("temp_id"))
)

# Join back to main dataframe
curr_df = silver_df.join(
    missing_invoices.select("InvoiceNo", "rep_customer_id"),
    on="InvoiceNo",
    how="left"
)

# Replace NULL CustomerIDs only
df_clean_customerID = curr_df.withColumn(
    "CustomerID",
    when(col("CustomerID").isNull(), col("rep_customer_id"))
    .otherwise(col("CustomerID"))
).drop("rep_customer_id")

# Count missing CustomerID after processing
missing_customer_count2 = df_clean_customerID.filter(col("CustomerID").isNull()).count()
print('Records with missing CustomerID (After Processing):', missing_customerID)

# Show top 10 rows with negative CustomerIDs
df_clean_customerID.filter(col("CustomerID") < 0).show(10, truncate=False)

Records with missing CustomerID (Before Processing): 135080
Records with missing CustomerID (After Processing): 135080
+---------+---------+---------------------------------+--------+-------------------+---------+----------+--------------+----+-----+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |year|month|
+---------+---------+---------------------------------+--------+-------------------+---------+----------+--------------+----+-----+
|536596   |21624    |VINTAGE UNION JACK DOORSTOP      |1       |2010-12-01 17:29:00|5.95     |-10016    |United Kingdom|2010|12   |
|536596   |22900    | SET 2 TEA TOWELS I LOVE LONDON  |1       |2010-12-01 17:29:00|2.95     |-10016    |United Kingdom|2010|12   |
|536596   |22114    |HOT WATER BOTTLE TEA AND SYMPATHY|1       |2010-12-01 17:29:00|3.95     |-10016    |United Kingdom|2010|12   |
|536596   |21967    |PACK OF 12 SKULL TISSUES         |1       |2010-12-01 17:29:00|0.29 

## Negative quantities (Returns & Cancellation Flags)

In [ ]:
# Identify & count return transactions (Quantity < 0)
return_count = df_clean_customerID.filter(col("Quantity") < 0).count()
print("Number of return transactions:", return_count)


# Add is_return flag column
silver_df = df_clean_customerID.withColumn(
    "is_return",
    when(col("Quantity") < 0, 1).otherwise(0)
)

# Show some rows with negative quantity (returns)
silver_df.filter(col("Quantity") < 0).select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "CustomerID"
).show(10, truncate=False)

Number of return transactions: 10624
+---------+---------+---------------------------------+--------+----------+
|InvoiceNo|StockCode|Description                      |Quantity|CustomerID|
+---------+---------+---------------------------------+--------+----------+
|C536379  |D        |Discount                         |-1      |14527     |
|C536543  |22632    |HAND WARMER RED RETROSPOT        |-1      |17841     |
|C536383  |35004C   |SET OF 3 COLOURED  FLYING DUCKS  |-1      |15311     |
|C536506  |22960    |JAM MAKING SET WITH JARS         |-6      |17897     |
|C536391  |22556    |PLASTERS IN TIN CIRCUS PARADE    |-12     |17548     |
|C536391  |21984    |PACK OF 12 PINK PAISLEY TISSUES  |-24     |17548     |
|C536391  |21983    |PACK OF 12 BLUE PAISLEY TISSUES  |-24     |17548     |
|C536391  |21980    |PACK OF 12 RED RETROSPOT TISSUES |-24     |17548     |
|C536391  |21484    |CHICK GREY HOT WATER BOTTLE      |-12     |17548     |
|C536391  |22557    |PLASTERS IN TIN VINTAGE PAISLE

## Cancelled Invoices

In [ ]:
# Identify & count cancelled invoices
cancelled_invoice_count = silver_df.filter(col("InvoiceNo").startswith("C")).count()
print("Number of cancelled invoice records:", cancelled_invoice_count)

# Cancellation flag
silver_df = silver_df.withColumn(
    "is_cancelled",
    when(col("InvoiceNo").startswith("C"), 1).otherwise(0)
)


# Cancellation type
silver_df = silver_df.withColumn(
    "cancellation_type",
    when(
        (col("is_cancelled") == 1) & (col("is_return") == 1),
        "Returned & Cancelled"
    ).when(
        col("is_cancelled") == 1,
        "Cancelled Only"
    ).otherwise("Normal Sale")
)

# Revenue Column
silver_df = silver_df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)
silver_df.show(10)

Number of cancelled invoice records: 9288
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+-----------------+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|year|month|is_return|is_cancelled|cancellation_type|Revenue|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+-----------------+-------+
|   536596|    21624|VINTAGE UNION JAC...|       1|2010-12-01 17:29:00|     5.95|    -10016|United Kingdom|2010|   12|        0|           0|      Normal Sale|   5.95|
|   536596|    22900| SET 2 TEA TOWELS...|       1|2010-12-01 17:29:00|     2.95|    -10016|United Kingdom|2010|   12|        0|           0|      Normal Sale|   2.95|
|   536596|    22114|HOT WATER BOTTLE ...|       1|2010-12-01 17:29:00|     3.95|    -10016|United Kingdom|2010|   12|

## Invalid or extreme prices

In [ ]:
# Invalid UnitPrice <= 0
df_valid_price = silver_df.filter(col("UnitPrice") > 0)

invalid_prices = silver_df.count() - df_valid_price.count()
print("Invalid price count:",invalid_prices)
print("Valid price count:",df_valid_price.count())



Invalid price count: 2521
Valid price count: 539388


In [ ]:
from pyspark.sql.functions import col, countDistinct, count

# Filter qty = 1 first
df_q1 = df_valid_price.filter(col("Quantity") == 1)

# Step 1: Find groups having multiple prices
multi_price = df_q1.groupBy(
    "StockCode","Description","InvoiceDate","Country"
).agg(
    countDistinct("UnitPrice").alias("price_types")
).filter(
    col("price_types") > 1
)

# Step 2: Join back to get counts per price
final_df = df_q1.join(
    multi_price,
    ["StockCode","Description","InvoiceDate","Country"],
    "inner"
).groupBy(
    "StockCode","Description","InvoiceDate","Country","UnitPrice"
).agg(
    count("*").alias("price_count")
).orderBy(
    "StockCode","InvoiceDate","UnitPrice","Country"
)
final_df.show(10, False)

+---------+-----------------------------------+-------------------+--------------+---------+-----------+
|StockCode|Description                        |InvoiceDate        |Country       |UnitPrice|price_count|
+---------+-----------------------------------+-------------------+--------------+---------+-----------+
|20914    |SET/5 RED RETROSPOT LID GLASS BOWLS|2010-12-20 15:03:00|United Kingdom|2.95     |1          |
|20914    |SET/5 RED RETROSPOT LID GLASS BOWLS|2010-12-20 15:03:00|United Kingdom|5.91     |1          |
|20969    |RED FLORAL FELTCRAFT SHOULDER BAG  |2011-11-24 09:26:00|United Kingdom|3.75     |1          |
|20969    |RED FLORAL FELTCRAFT SHOULDER BAG  |2011-11-24 09:26:00|United Kingdom|7.46     |1          |
|21181    |PLEASE ONE PERSON METAL SIGN       |2010-12-20 10:14:00|United Kingdom|2.09     |1          |
|21181    |PLEASE ONE PERSON METAL SIGN       |2010-12-20 10:14:00|United Kingdom|4.21     |1          |
|21238    |RED RETROSPOT CUP                  |2010-12-

In [ ]:
from pyspark.sql.functions import collect_set, col, size


# Price Inconsistency Check
price_details = df_valid_price \
    .filter(col("Quantity") == 1) \
    .groupBy("StockCode", "Description","InvoiceDate") \
    .agg(collect_set("UnitPrice").alias("price_list")) \
    .filter(size(col("price_list")) > 1)

price_details.show(10, False)

+---------+----------------------------------+-------------------+-------------+
|StockCode|Description                       |InvoiceDate        |price_list   |
+---------+----------------------------------+-------------------+-------------+
|20969    |RED FLORAL FELTCRAFT SHOULDER BAG |2011-11-24 09:26:00|[3.75, 7.46] |
|21181    |PLEASE ONE PERSON METAL SIGN      |2010-12-20 10:14:00|[2.09, 4.21] |
|21238    |RED RETROSPOT CUP                 |2010-12-21 15:40:00|[1.66, 2.51] |
|21481    |FAWN BLUE HOT WATER BOTTLE        |2011-12-09 10:23:00|[8.29, 10.29]|
|21485    |RETROSPOT HEART HOT WATER BOTTLE  |2011-08-10 09:29:00|[12.21, 7.46]|
|21524    |DOORMAT SPOTTY HOME SWEET HOME    |2010-12-10 14:59:00|[7.95, 14.43]|
|21774    |DECORATIVE CATS BATHROOM BOTTLE   |2011-01-21 17:09:00|[2.46, 1.25] |
|21868    |POTTING SHED TEA MUG              |2010-12-10 14:59:00|[1.24, 3.36] |
|22328    |ROUND SNACK BOXES SET OF 4 FRUITS |2010-12-21 15:40:00|[6.77, 4.21] |
|22371    |AIRLINE BAG VINTA

In [ ]:
from pyspark.sql.functions import col, when


# Calculate IQR for UnitPrice
quantiles = silver_df.approxQuantile("UnitPrice", [0.25, 0.75], 0.01)

Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

# Detect Outliers (IQR + Business Rule)
price_outliers = silver_df.filter(
    (col("UnitPrice") <= 0) |          # Business rule
    (col("UnitPrice") < lower_bound) | # IQR lower
    (col("UnitPrice") > upper_bound)   # IQR upper
)

price_outlier_count = price_outliers.count()

print("Total Price Outliers:", price_outlier_count)

price_outliers.show(10, truncate=False)


# Add Outlier Flag Column

silver_df = silver_df.withColumn(
    "is_price_outlier",
    when(
        (col("UnitPrice") <= 0) |
        (col("UnitPrice") < lower_bound) |
        (col("UnitPrice") > upper_bound),
        1
    ).otherwise(0)
)

silver_df.groupBy("is_price_outlier").count().show()

Q1: 1.25
Q3: 4.13
Lower Bound: -3.0700000000000003
Upper Bound: 8.45
Total Price Outliers: 42146
+---------+---------+--------------------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+-----------------+-------+
|InvoiceNo|StockCode|Description                     |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |year|month|is_return|is_cancelled|cancellation_type|Revenue|
+---------+---------+--------------------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+-----------------+-------+
|536596   |22802    |FAUX FUR CHOCOLATE THROW        |1       |2010-12-01 17:29:00|19.95    |-10016    |United Kingdom|2010|12   |0        |0           |Normal Sale      |19.95  |
|536938   |21258    |VICTORIAN SEWING BOX LARGE      |24      |2010-12-03 12:05:00|10.95    |14680     |United Kingdom|2010|12   |0        |0           |Normal Sale      |262.80 |
|53

In [ ]:
# Price outliers
zero_or_negative_price = silver_df.filter(col("UnitPrice") <= 0).count()
extremely_high_price = silver_df.filter(col("UnitPrice") > 100).count()

print("UnitPrice <= 0 (Invalid Price):", zero_or_negative_price)
print("UnitPrice > 100 (Extreme Price):", extremely_high_price)

UnitPrice <= 0 (Invalid Price): 2521
UnitPrice > 100 (Extreme Price): 1036


In [ ]:
# Get Q1 and Q3 for UnitPrice
quantiles = silver_df.approxQuantile("UnitPrice", [0.25, 0.75], 0.01)

Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("UnitPrice Lower Bound:", lower_bound)
print("UnitPrice Upper Bound:", upper_bound)

# Detect price outliers
price_outliers = silver_df.filter(
    (col("UnitPrice") < lower_bound) |
    (col("UnitPrice") > upper_bound)
)

print("Total Price Outliers:", price_outliers.count())

price_outliers.show(10, truncate=False)

UnitPrice Lower Bound: -3.0700000000000003
UnitPrice Upper Bound: 8.45
Total Price Outliers: 39627
+---------+---------+--------------------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+-----------------+-------+----------------+
|InvoiceNo|StockCode|Description                     |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |year|month|is_return|is_cancelled|cancellation_type|Revenue|is_price_outlier|
+---------+---------+--------------------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+-----------------+-------+----------------+
|536596   |22802    |FAUX FUR CHOCOLATE THROW        |1       |2010-12-01 17:29:00|19.95    |-10016    |United Kingdom|2010|12   |0        |0           |Normal Sale      |19.95  |1               |
|536938   |21258    |VICTORIAN SEWING BOX LARGE      |24      |2010-12-03 12:05:00|10.95    |14680     |United Ki

## Remove Duplicates

In [ ]:
# Drop duplicate invoice lines
df_deduplicated = df_valid_price.dropDuplicates(["InvoiceNo", "StockCode", "quantity","InvoiceDate", "UnitPrice","Country"])

removed_duplicates = df_valid_price.count() - df_deduplicated.count()
print("Records removed due to duplicates:", removed_duplicates)
print("Final record count:", df_deduplicated.count())

# Show some rows
df_deduplicated.show(10, truncate=False)

Records removed due to duplicates: 5265
Final record count: 534123
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+--------------------+--------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |year|month|is_return|is_cancelled|cancellation_type   |Revenue |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+------------+--------------------+--------+
|C536757  |84347    |ROTATING SILVER ANGELS T-LIGHT HLDR|-9360   |2010-12-02 14:23:00|0.03     |15838     |United Kingdom|2010|12   |1        |1           |Returned & Cancelled|-280.80 |
|C570556  |20971    |PINK BLUE FELT CRAFT TRINKET BOX   |-1296   |2011-10-11 11:10:00|1.06     |16029     |United Kingdom|2011|10   |1        |1           |Returned & Cancelled|-1373.76

## Check Null Counts after cleaning

In [ ]:
from pyspark.sql.functions import sum as spark_sum

null_counts1 = df_deduplicated.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_deduplicated.columns
])

null_counts1.show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+----+-----+---------+------------+-----------------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|year|month|is_return|is_cancelled|cancellation_type|Revenue|
+---------+---------+-----------+--------+-----------+---------+----------+-------+----+-----+---------+------------+-----------------+-------+
|        0|        0|          0|       0|          0|        0|         0|      0|   0|    0|        0|           0|                0|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+----+-----+---------+------------+-----------------+-------+



## Data Quality Report

In [ ]:
from pyspark.sql.functions import lit

data_quality_report = spark.createDataFrame([
    ("Total Records", total_records),
    ("Missing CustomerID (replaced)", missing_records),
    ("Return Transactions", return_count),
    ("Cancelled Invoices", cancelled_invoice_count),
    ("Invalid/Negative Prices Removed", invalid_prices),
    ("Duplicate Records Removed", removed_duplicates),
], ["Rule", "Count"])

data_quality_report.show(truncate=False)

+-------------------------------+------+
|Rule                           |Count |
+-------------------------------+------+
|Total Records                  |541909|
|Missing CustomerID (replaced)  |135080|
|Return Transactions            |10624 |
|Cancelled Invoices             |9288  |
|Invalid/Negative Prices Removed|2521  |
|Duplicate Records Removed      |5265  |
+-------------------------------+------+



## Write Silver Layer

In [ ]:
silver_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Big Data/Assignment 02/silver_parquet"
)

print("✅ Silver layer saved at /content/drive/MyDrive/Big Data/Assignment 02/silver_parquet")

✅ Silver layer saved at /content/drive/MyDrive/Big Data/Assignment 02/silver_parquet


# 🔵 Feature Engineering

In [ ]:
silver_df = spark.read.parquet("/content/drive/MyDrive/Big Data/Assignment 02/silver_parquet")

## Revenue Features

In [ ]:
from pyspark.sql.functions import col

df_features = silver_df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)
df_features.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+---------+----+-----+---------+------------+-----------------+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|  Country|year|month|is_return|is_cancelled|cancellation_type|Revenue|
+---------+---------+--------------------+--------+-------------------+---------+----------+---------+----+-----+---------+------------+-----------------+-------+
|   556917|   16161P|  WRAP ENGLISH ROSE |     400|2011-06-15 13:37:00|     0.34|     12415|Australia|2011|    6|        0|           0|      Normal Sale| 136.00|
|   547659|    20665|RED RETROSPOT PURSE |       6|2011-03-24 13:05:00|     2.95|     12434|Australia|2011|    3|        0|           0|      Normal Sale|  17.70|
|   540267|    20675|  BLUE POLKADOT BOWL|      72|2011-01-06 11:12:00|     1.06|     12415|Australia|2011|    1|        0|           0|      Normal Sale|  76.32|
|   540267|    20676| 

## Time-based features (hour, weekday, month)

In [ ]:
from pyspark.sql.functions import hour, dayofweek, month

df_features = df_features \
    .withColumn("hour", hour("InvoiceDate")) \
    .withColumn("weekday", dayofweek("InvoiceDate")) \
    .withColumn("month", month("InvoiceDate"))

df_features.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+---------+----+-----+---------+------------+-----------------+-------+----+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|  Country|year|month|is_return|is_cancelled|cancellation_type|Revenue|hour|weekday|
+---------+---------+--------------------+--------+-------------------+---------+----------+---------+----+-----+---------+------------+-----------------+-------+----+-------+
|   556917|   16161P|  WRAP ENGLISH ROSE |     400|2011-06-15 13:37:00|     0.34|     12415|Australia|2011|    6|        0|           0|      Normal Sale| 136.00|  13|      4|
|   547659|    20665|RED RETROSPOT PURSE |       6|2011-03-24 13:05:00|     2.95|     12434|Australia|2011|    3|        0|           0|      Normal Sale|  17.70|  13|      5|
|   540267|    20675|  BLUE POLKADOT BOWL|      72|2011-01-06 11:12:00|     1.06|     12415|Australia|2011|    1|       

## Basket-level metrics

In [ ]:
from pyspark.sql.functions import sum, countDistinct

basket_metrics = df_features.groupBy("InvoiceNo").agg(
    sum("Quantity").alias("BasketQty"),
    sum("Revenue").alias("BasketRevenue"),
    countDistinct("StockCode").alias("UniqueItems")
)

basket_metrics.show(10)

+---------+---------+-------------+-----------+
|InvoiceNo|BasketQty|BasketRevenue|UniqueItems|
+---------+---------+-------------+-----------+
|   541518|     2334|      2919.51|        101|
|   576059|      123|       357.67|         42|
|   566431|      254|       303.36|         18|
|   578057|      128|       162.78|         27|
|   580906|      108|       339.48|          4|
|   563020|      385|       605.14|         24|
|   574844|       89|       222.26|         13|
|   550617|      191|       376.00|         21|
|   540499|       88|       348.20|         22|
|   542026|       69|       169.65|          9|
+---------+---------+-------------+-----------+
only showing top 10 rows


## Customer RFM features

In [ ]:
from pyspark.sql.functions import datediff, current_date, max, count

customer_rfm = df_features.groupBy("CustomerID").agg(
    sum("Revenue").alias("Monetary"),
    count("InvoiceNo").alias("Frequency"),
    datediff(current_date(), max("InvoiceDate")).alias("Recency")
)

customer_rfm .show(5)

+----------+--------+---------+-------+
|CustomerID|Monetary|Frequency|Recency|
+----------+--------+---------+-------+
|     16503| 1421.43|       86|   5292|
|     13623|  652.40|       83|   5216|
|     16386|  302.57|       81|   5214|
|     15727| 5159.06|      301|   5202|
|     13285| 2709.12|      187|   5209|
+----------+--------+---------+-------+
only showing top 5 rows


## Spark Window-based feature

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

windowSpec = Window.partitionBy("CustomerID") \
                   .orderBy(col("Monetary").desc())

customer_rank = customer_rfm.withColumn(
    "customer_rank",
    rank().over(windowSpec)
)

customer_rank.show(10)

+----------+---------+---------+-------+-------------+
|CustomerID| Monetary|Frequency|Recency|customer_rank|
+----------+---------+---------+-------+-------------+
|    -13710|-17836.46|        1|   5190|            1|
|    -13709|-11586.50|        1|   5190|            1|
|    -13708|   -15.30|        1|   5194|            1|
|    -13707|   -15.60|        1|   5195|            1|
|    -13706| -1208.04|        7|   5203|            1|
|    -13705|  -288.00|        3|   5204|            1|
|    -13704| -1149.98|        1|   5207|            1|
|    -13703|  -530.25|        1|   5207|            1|
|    -13702|   -27.21|        1|   5207|            1|
|    -13701|  -490.06|        1|   5207|            1|
+----------+---------+---------+-------+-------------+
only showing top 10 rows


## Product Level Feature

In [ ]:
product_features = df_features.groupBy("StockCode").agg(
    sum("Quantity").alias("Total_Units_Sold"),
    sum("Revenue").alias("Total_Revenue"),
    countDistinct("InvoiceNo").alias("Total_Orders")
)

product_features.show(10, truncate=False)

+---------+----------------+-------------+------------+
|StockCode|Total_Units_Sold|Total_Revenue|Total_Orders|
+---------+----------------+-------------+------------+
|22728    |5315            |20518.40     |799         |
|21249    |724             |2160.00      |119         |
|21259    |1163            |6985.00      |293         |
|21452    |789             |2468.91      |196         |
|21889    |6364            |8814.79      |589         |
|22121    |351             |2153.75      |134         |
|23318    |3641            |8793.70      |383         |
|22596    |3451            |4079.70      |268         |
|90143    |31              |237.18       |20          |
|23459    |30              |375.00       |21          |
+---------+----------------+-------------+------------+
only showing top 10 rows


## Customer Level Feature

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import datediff, current_date, col

customer_features = df_features.groupBy("CustomerID").agg(
    F.min("InvoiceDate").alias("first_purchase_date"),
    F.max("InvoiceDate").alias("last_purchase_date"),
    F.countDistinct("InvoiceNo").alias("frequency"),
    F.sum("Revenue").alias("monetary"),
    F.avg("Revenue").alias("avg_order_value")
).withColumn(
    "recency_days",
    datediff(current_date(), col("last_purchase_date"))
)

customer_features.show(10, truncate=False)

+----------+-------------------+-------------------+---------+--------+---------------+------------+
|CustomerID|first_purchase_date|last_purchase_date |frequency|monetary|avg_order_value|recency_days|
+----------+-------------------+-------------------+---------+--------+---------------+------------+
|12940     |2011-09-13 10:16:00|2011-10-24 14:04:00|4        |862.44  |8.624400       |5232        |
|-13700    |2011-11-18 14:16:00|2011-11-18 14:16:00|1        |-83.88  |-27.960000     |5207        |
|15727     |2010-12-15 15:47:00|2011-11-23 12:36:00|7        |5159.06 |17.139734      |5202        |
|15790     |2011-11-29 14:53:00|2011-11-29 14:53:00|1        |218.75  |6.433824       |5196        |
|13285     |2011-02-22 17:13:00|2011-11-16 13:19:00|4        |2709.12 |14.487273      |5209        |
|13832     |2011-11-20 15:36:00|2011-11-22 12:31:00|2        |40.95   |10.237500      |5203        |
|17389     |2011-01-04 17:12:00|2011-12-09 09:38:00|43       |31300.08|139.732500     |5186

## Discount/Promotion features

In [ ]:
from pyspark.sql.functions import col, when, round
from pyspark.sql import functions as F

# Create a promo flag: 1 if revenue > 100, else 0
# Use df_deduplicated as it contains the 'Revenue' column
df_with_promo_flag = df_features.withColumn(
    "promo_flag",
    when(col("Revenue") > 100, 1).otherwise(0)
)

# Customer-level promo metrics
promo_features = df_with_promo_flag.groupBy("CustomerID").agg(
    F.sum("promo_flag").alias("total_high_value_transactions"),
    round(
      (F.sum("promo_flag") / F.count("InvoiceNo") * 100),
      2
    ).alias("high_value_transaction_pct")
)

promo_features.show(250, truncate=False)

+----------+-----------------------------+--------------------------+
|CustomerID|total_high_value_transactions|high_value_transaction_pct|
+----------+-----------------------------+--------------------------+
|16503     |0                            |0.0                       |
|13623     |1                            |1.2                       |
|16386     |0                            |0.0                       |
|15727     |2                            |0.66                      |
|13285     |0                            |0.0                       |
|15957     |0                            |0.0                       |
|16339     |0                            |0.0                       |
|15790     |0                            |0.0                       |
|18024     |1                            |4.55                      |
|15447     |0                            |0.0                       |
|17389     |100                          |44.64                     |
|-12070    |0       

## Engagement features

In [ ]:
from pyspark.sql import functions as F

engagement_features = df_features.groupBy("CustomerID").agg(
    F.countDistinct("InvoiceNo").alias("transactions_per_channel")
)

engagement_features.show(10, truncate=False)

+----------+------------------------+
|CustomerID|transactions_per_channel|
+----------+------------------------+
|12940     |4                       |
|-13700    |1                       |
|15727     |7                       |
|15790     |1                       |
|13285     |4                       |
|13832     |2                       |
|17389     |43                      |
|15957     |1                       |
|16503     |5                       |
|15447     |1                       |
+----------+------------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import sum, avg, countDistinct

country_features = df_features.groupBy("Country").agg(
    sum("Revenue").alias("Country_Total_Revenue"),
    sum("Quantity").alias("Country_Total_Quantity"),
    avg("UnitPrice").alias("Country_Avg_Unit_Price"),
    countDistinct("CustomerID").alias("Country_Customer_Count")
)

country_features.show()

+------------------+---------------------+----------------------+----------------------+----------------------+
|           Country|Country_Total_Revenue|Country_Total_Quantity|Country_Avg_Unit_Price|Country_Customer_Count|
+------------------+---------------------+----------------------+----------------------+----------------------+
|            Sweden|             36585.41|                 35632|              3.914816|                     8|
|         Singapore|              9120.39|                  5234|            109.645808|                     1|
|           Germany|            221509.47|                117339|              3.970610|                    95|
|               RSA|              1002.31|                   351|              4.352632|                     1|
|            France|            197317.11|                110437|              5.033487|                    90|
|            Greece|              4710.52|                  1556|              4.885548|                

In [ ]:
gold_df = df_features

gold_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Big Data/Assignment 02/gold_parquet"
)

In [ ]:
gold_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_return: integer (nullable = true)
 |-- is_cancelled: integer (nullable = true)
 |-- cancellation_type: string (nullable = true)
 |-- Revenue: decimal(21,2) (nullable = true)
 |-- hour: integer (nullable = true)
 |-- weekday: integer (nullable = true)



# 🔵 MongoDB Data Modeling

In [ ]:
!pip install pymongo

from urllib.parse import quote_plus
from pymongo import MongoClient

"""
username = quote_plus("YourUserName")
password = quote_plus("YourPassword")
host = "##########"
db_name = "DatabaseName"
"""

db_name = "DatabaseName"
uri = "mongodb+srv:/###############################.mongodb.net/"

client = MongoClient(uri)
db = client[db_name]

print("✅ MongoDB Connected")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.1 MB/s eta 0:00:00
✅ MongoDB Connected


In [ ]:
from datetime import datetime
import pandas as pd # Import pandas

fact_rows = []

# Convert PySpark DataFrame to Pandas DataFrame for iterrows()
pandas_gold_df = gold_df.toPandas()

for _, r in pandas_gold_df.iterrows():

    invoice_date = r["InvoiceDate"]

    if isinstance(invoice_date, str):
        invoice_date = datetime.strptime(invoice_date, "%Y-%m-%d %H:%M")

    fact_rows.append({

        "InvoiceNo": str(r["InvoiceNo"]),
        "StockCode": str(r["StockCode"]),
        "Description": r["Description"],
        "Quantity": int(r["Quantity"]),
        "InvoiceDate": invoice_date,
        "UnitPrice": float(r["UnitPrice"]),
        "CustomerID": int(r["CustomerID"]) if not pd.isna(r["CustomerID"]) else None,
        "Country": r["Country"],

        # Time dimensions
        "year": invoice_date.year,
        "month": invoice_date.month,
        "hour": invoice_date.hour,
        "weekday": invoice_date.weekday(),

        # Flags
        "is_return": 1 if r["Quantity"] < 0 else 0,
        "is_cancelled": 1 if str(r["InvoiceNo"]).startswith("C") else 0,

        "cancellation_type":
            "Return" if r["Quantity"] < 0 else
            "Cancelled" if str(r["InvoiceNo"]).startswith("C") else
            "Normal",

        "Revenue": float(r["Revenue"])
    })

# Insert

db.fact_invoices.delete_many({})
db.fact_invoices.insert_many(fact_rows)

print("✅ fact_invoices recreated with correct schema")

✅ fact_invoices recreated with correct schema


In [ ]:
dim_customers = []

for row in customer_rfm.collect():

    monetary = float(row["Monetary"])  # Convert Decimal to float

    if monetary > 5000:
        segment = "High Value"
    elif monetary > 1000:
        segment = "Medium Value"
    else:
        segment = "Low Value"

    doc = row.asDict()
    doc["Monetary"] = monetary # Update the dictionary with the float value
    doc["segment"] = segment

    dim_customers.append(doc)

db.dim_customers.delete_many({})
db.dim_customers.insert_many(dim_customers)

print("✅ dim_customers created")

✅ dim_customers created


In [ ]:
product_perf = {}

for r in silver_df.collect():

    code = r["StockCode"]

    if code not in product_perf:
        product_perf[code] = {
            "StockCode": code,
            "Description": r["Description"],
            "total_units_sold": 0,
            "total_revenue": 0,
            "total_orders": set(),
            "price_sum": 0,
            "price_count": 0,
            "countries": {}
        }

    p = product_perf[code]

    p["total_units_sold"] += int(r["Quantity"])
    p["total_revenue"] += float(r["Revenue"])
    p["total_orders"].add(r["InvoiceNo"])
    p["price_sum"] += float(r["UnitPrice"])
    p["price_count"] += 1

    country = r["Country"]

    if country not in p["countries"]:
        p["countries"][country] = {"units": 0, "revenue": 0}

    p["countries"][country]["units"] += int(r["Quantity"])
    p["countries"][country]["revenue"] += float(r["Revenue"])

In [ ]:
dim_products = []

for p in product_perf.values():

    country_list = []

    for c, v in p["countries"].items():
        country_list.append({
            "Country": c,
            "units_sold_country": v["units"],
            "revenue_country": v["revenue"]
        })

    dim_products.append({
        "StockCode": p["StockCode"],
        "Description": p["Description"],
        "total_units_sold": p["total_units_sold"],
        "total_revenue": p["total_revenue"],
        "avg_unit_price": p["price_sum"] / p["price_count"],
        "total_orders": len(p["total_orders"]),
        "country_distribution": country_list
    })

In [ ]:
db.dim_products.delete_many({})
db.dim_products.insert_many(dim_products)

print("✅ dim_products created")

✅ dim_products created


# 🔵 MongoDB Indexing & Write Optimization

## Create indexes

In [ ]:
db.fact_invoices.create_index([("InvoiceNo",1)])
db.fact_invoices.create_index([("InvoiceTotalRevenue",-1)])

db.dim_customers.create_index([("CustomerID",1)])
db.dim_customers.create_index([("segment",1)])

db.dim_products.create_index([("StockCode",1)])

print("✅ Indexes Created")

✅ Indexes Created


## Show Index Information

In [ ]:
print("\n fact_invoices Indexes")
print(db.fact_invoices.index_information())

print("\n dim_customers Indexes")
print(db.dim_customers.index_information())

print("\n dim_products Indexes")
print(db.dim_products.index_information())


 fact_invoices Indexes
{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'InvoiceNo_1': {'v': 2, 'key': [('InvoiceNo', 1)]}, 'InvoiceTotalRevenue_-1': {'v': 2, 'key': [('InvoiceTotalRevenue', -1)]}}

 dim_customers Indexes
{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'CustomerID_1': {'v': 2, 'key': [('CustomerID', 1)]}, 'segment_1': {'v': 2, 'key': [('segment', 1)]}}

 dim_products Indexes
{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'StockCode_1': {'v': 2, 'key': [('StockCode', 1)]}}


## Query Performance Test

In [ ]:
import time
print("\n Performance Test Started")

# Query BEFORE / AFTER index (run once before creating index if needed)

start = time.time()
results = list( db.fact_invoices.find( {"InvoiceTotalRevenue": {"$gt": 5000}} ) )

end = time.time()

print("Query Time:", end - start)


 Performance Test Started
Query Time: 2.3247554302215576


In [ ]:
explain_plan = db.fact_invoices.find(
    {"InvoiceTotalRevenue": {"$gt": 5000}}
).explain()

print("\n📊 Explain Plan Stage:")
print(explain_plan["executionStats"]["executionStages"]["stage"])
print("\n✅ Task 05 Completed Successfully")


📊 Explain Plan Stage:
FETCH

✅ Task 05 Completed Successfully


# 🔵 Analytics & Insights

## Create Gold Dataset

In [ ]:
gold_df = df_features

gold_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Big Data/Assignment 02/gold_parquet"
)

In [ ]:
gold_df = spark.read.parquet("/content/drive/MyDrive/Big Data/Assignment 02/gold_parquet")

In [ ]:
gold_df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+---------+----+-----+---------+------------+-----------------+-------+----+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|  Country|year|month|is_return|is_cancelled|cancellation_type|Revenue|hour|weekday|
+---------+---------+--------------------+--------+-------------------+---------+----------+---------+----+-----+---------+------------+-----------------+-------+----+-------+
|   556917|   16161P|  WRAP ENGLISH ROSE |     400|2011-06-15 13:37:00|     0.34|     12415|Australia|2011|    6|        0|           0|      Normal Sale| 136.00|  13|      4|
|   547659|    20665|RED RETROSPOT PURSE |       6|2011-03-24 13:05:00|     2.95|     12434|Australia|2011|    3|        0|           0|      Normal Sale|  17.70|  13|      5|
|   540267|    20675|  BLUE POLKADOT BOWL|      72|2011-01-06 11:12:00|     1.06|     12415|Australia|2011|    1|       

## Part A - Spark-based analytics queries

## Monthly Revenue

In [ ]:
from pyspark.sql.functions import year, month, sum

monthly_revenue = gold_df \
    .groupBy(year("InvoiceDate").alias("Year"),
             month("InvoiceDate").alias("Month")) \
    .agg(sum("Revenue").alias("MonthlyRevenue"))

monthly_revenue.show()

+----+-----+--------------+
|Year|Month|MonthlyRevenue|
+----+-----+--------------+
|2010|   12|     746723.61|
|2011|    3|     682013.98|
|2011|    2|     497026.41|
|2011|    4|     492367.84|
|2011|   12|     432701.06|
|2011|    8|     703510.58|
|2011|    5|     722079.25|
|2011|    7|     680156.99|
|2011|    6|     689977.23|
|2011|   11|    1455608.80|
|2011|   10|    1069368.23|
|2011|    1|     558448.56|
|2011|    9|    1017596.68|
+----+-----+--------------+



## Top customers by spend

In [ ]:
top_customers = gold_df.groupBy("CustomerID") \
    .agg(sum("Revenue").alias("TotalSpend")) \
    .orderBy(col("TotalSpend").desc())

top_customers.show(10)

+----------+----------+
|CustomerID|TotalSpend|
+----------+----------+
|     14646| 279489.02|
|     18102| 256438.49|
|     17450| 187322.17|
|     14911| 132458.73|
|     12415| 123725.45|
|     14156| 113214.59|
|     17511|  88125.38|
|     16684|  65892.08|
|     13694|  62690.54|
|     15311|  59284.19|
+----------+----------+
only showing top 10 rows


## Top Products

In [ ]:
top_products = gold_df.groupBy("StockCode") \
    .agg(sum("Revenue").alias("ProductRevenue")) \
    .orderBy(col("ProductRevenue").desc())

top_products.show(10)

+---------+--------------+
|StockCode|ProductRevenue|
+---------+--------------+
|      DOT|     206245.48|
|    22423|     164459.49|
|    47566|      98243.88|
|   85123A|      97838.45|
|   85099B|      92175.79|
|    23084|      66661.63|
|     POST|      66230.64|
|    22086|      63715.24|
|    84879|      58792.42|
|    79321|      53746.66|
+---------+--------------+
only showing top 10 rows


## Country Sales

In [ ]:
country_sales = gold_df.groupBy("Country") \
    .agg(sum("Revenue").alias("CountryRevenue"))


country_sales.show()

+------------------+--------------+
|           Country|CountryRevenue|
+------------------+--------------+
|            Sweden|      36585.41|
|         Singapore|       9120.39|
|           Germany|     221509.47|
|               RSA|       1002.31|
|            France|     197317.11|
|            Greece|       4710.52|
|European Community|       1291.75|
|           Belgium|      40910.96|
|           Finland|      22326.74|
|             Malta|       2505.47|
|       Unspecified|       4740.94|
|             Italy|      16890.51|
|              EIRE|     262993.38|
|         Lithuania|       1661.06|
|            Norway|      35163.46|
|             Spain|      54756.03|
|           Denmark|      18768.14|
|         Hong Kong|       9908.24|
|           Iceland|       4310.00|
|            Israel|       7901.97|
+------------------+--------------+
only showing top 20 rows


## Returns Analysis

In [ ]:
returns = gold_df.filter(col("Quantity") < 0) \
    .groupBy("Country") \
    .count()

returns.show()

+---------------+-----+
|        Country|count|
+---------------+-----+
|         Sweden|   11|
|        Germany|  453|
|         France|  148|
|        Belgium|   38|
|        Finland|   10|
|          Malta|   15|
|          Italy|   45|
|           EIRE|  301|
|         Norway|   14|
|          Spain|   48|
|        Denmark|    9|
|      Hong Kong|    4|
|         Israel|    2|
|Channel Islands|   10|
|         Cyprus|    8|
|    Switzerland|   35|
| Czech Republic|    5|
|          Japan|   37|
|         Poland|   11|
|       Portugal|   18|
+---------------+-----+
only showing top 20 rows


## Part B - MongoDB aggregation pipelines

## Monthly revenue

In [ ]:
list(db.fact_invoices.aggregate([
 {
  "$group":{
   "_id":{"year":"$year","month":"$month"},
   "MonthlyRevenue":{"$sum":"$Revenue"}
  }
 },
 {"$sort":{"_id.year":1,"_id.month":1}}
]))

[{'_id': {'year': 2010, 'month': 12}, 'MonthlyRevenue': 746723.61},
 {'_id': {'year': 2011, 'month': 1}, 'MonthlyRevenue': 558448.5599999999},
 {'_id': {'year': 2011, 'month': 2}, 'MonthlyRevenue': 497026.41},
 {'_id': {'year': 2011, 'month': 3}, 'MonthlyRevenue': 682013.98},
 {'_id': {'year': 2011, 'month': 4}, 'MonthlyRevenue': 492367.84},
 {'_id': {'year': 2011, 'month': 5}, 'MonthlyRevenue': 722079.25},
 {'_id': {'year': 2011, 'month': 6}, 'MonthlyRevenue': 689977.23},
 {'_id': {'year': 2011, 'month': 7}, 'MonthlyRevenue': 680156.99},
 {'_id': {'year': 2011, 'month': 8}, 'MonthlyRevenue': 703510.58},
 {'_id': {'year': 2011, 'month': 9}, 'MonthlyRevenue': 1017596.68},
 {'_id': {'year': 2011, 'month': 10}, 'MonthlyRevenue': 1069368.23},
 {'_id': {'year': 2011, 'month': 11}, 'MonthlyRevenue': 1455608.8},
 {'_id': {'year': 2011, 'month': 12}, 'MonthlyRevenue': 432701.06}]

## Top customers

In [ ]:
pipeline = [
    {
        "$match": {
            "InvoiceNo": { "$not": { "$regex": "^C" } }
        }
    },
    {
        "$group": {
            "_id": "$CustomerID",
            "TotalSpend": { "$sum": "$Revenue" }
        }
    },
    { "$sort": { "TotalSpend": -1 } },
    { "$limit": 10 }
]

result = db.fact_invoices.aggregate(pipeline)

for doc in result:
    print(doc)

{'_id': 14646, 'TotalSpend': 280206.02}
{'_id': 18102, 'TotalSpend': 259657.3}
{'_id': 17450, 'TotalSpend': 194390.79}
{'_id': 16446, 'TotalSpend': 168472.5}
{'_id': 14911, 'TotalSpend': 143711.17}
{'_id': 12415, 'TotalSpend': 124914.53}
{'_id': 14156, 'TotalSpend': 117210.08}
{'_id': 17511, 'TotalSpend': 91062.38}
{'_id': 16029, 'TotalSpend': 80850.84}
{'_id': 12346, 'TotalSpend': 77183.6}


## Top products

In [ ]:
pipeline = [
    {
        "$match": {
            "InvoiceNo": { "$not": { "$regex": "^C" } }
        }
    },
    {
        "$group": {
            "_id": "$StockCode",
            "ProductRevenue": { "$sum": "$Revenue" }
        }
    },
    { "$sort": { "ProductRevenue": -1 } },
    { "$limit": 10 }
]

result = db.fact_invoices.aggregate(pipeline)

for doc in result:
    print(doc)

{'_id': 'DOT', 'ProductRevenue': 206248.77}
{'_id': '22423', 'ProductRevenue': 174156.54}
{'_id': '23843', 'ProductRevenue': 168469.6}
{'_id': '85123A', 'ProductRevenue': 104462.75}
{'_id': '47566', 'ProductRevenue': 99445.23}
{'_id': '85099B', 'ProductRevenue': 94159.81}
{'_id': '23166', 'ProductRevenue': 81700.92000000001}
{'_id': 'POST', 'ProductRevenue': 78101.88}
{'_id': 'M', 'ProductRevenue': 77750.27}
{'_id': '23084', 'ProductRevenue': 66870.03}


## Country sales

In [ ]:
pipeline = [
    {
        "$match": {
            "InvoiceNo": { "$not": { "$regex": "^C" } }
        }
    },
    {
        "$group": {
            "_id": "$Country",
            "CountryRevenue": { "$sum": "$Revenue" }
        }
    },
    { "$sort": { "CountryRevenue": -1 } }
]

result = db.fact_invoices.aggregate(pipeline)

for doc in result:
    print(doc)

{'_id': 'United Kingdom', 'CountryRevenue': 9001192.24}
{'_id': 'Netherlands', 'CountryRevenue': 285446.34}
{'_id': 'EIRE', 'CountryRevenue': 283140.52}
{'_id': 'Germany', 'CountryRevenue': 228678.4}
{'_id': 'France', 'CountryRevenue': 209625.37}
{'_id': 'Australia', 'CountryRevenue': 138453.81}
{'_id': 'Spain', 'CountryRevenue': 61558.56}
{'_id': 'Switzerland', 'CountryRevenue': 57067.6}
{'_id': 'Belgium', 'CountryRevenue': 41196.34}
{'_id': 'Sweden', 'CountryRevenue': 38367.83}
{'_id': 'Japan', 'CountryRevenue': 37416.37}
{'_id': 'Norway', 'CountryRevenue': 36165.44}
{'_id': 'Portugal', 'CountryRevenue': 33683.05}
{'_id': 'Finland', 'CountryRevenue': 22546.08}
{'_id': 'Singapore', 'CountryRevenue': 21279.29}
{'_id': 'Channel Islands', 'CountryRevenue': 20440.54}
{'_id': 'Denmark', 'CountryRevenue': 18955.34}
{'_id': 'Italy', 'CountryRevenue': 17483.24}
{'_id': 'Hong Kong', 'CountryRevenue': 15483.0}
{'_id': 'Cyprus', 'CountryRevenue': 13502.85}
{'_id': 'Austria', 'CountryRevenue': 10

##  Return or cancellation

In [ ]:
pipeline = [
    {
        "$match": {
            "InvoiceNo": { "$regex": "^C" },   # Only cancelled invoices
            "Quantity": { "$lt": 0 }           # Returned items
        }
    },
    {
        "$group": {
            "_id": "$Country",
            "ReturnCount": { "$sum": 1 }
        }
    },
    { "$sort": { "ReturnCount": -1 } }
]

result = db.fact_invoices.aggregate(pipeline)

for doc in result:
    print(doc)

{'_id': 'United Kingdom', 'ReturnCount': 7821}
{'_id': 'Germany', 'ReturnCount': 453}
{'_id': 'EIRE', 'ReturnCount': 301}
{'_id': 'France', 'ReturnCount': 148}
{'_id': 'USA', 'ReturnCount': 112}
{'_id': 'Australia', 'ReturnCount': 74}
{'_id': 'Spain', 'ReturnCount': 48}
{'_id': 'Italy', 'ReturnCount': 45}
{'_id': 'Belgium', 'ReturnCount': 38}
{'_id': 'Japan', 'ReturnCount': 37}
{'_id': 'Switzerland', 'ReturnCount': 35}
{'_id': 'Portugal', 'ReturnCount': 18}
{'_id': 'Malta', 'ReturnCount': 15}
{'_id': 'Norway', 'ReturnCount': 14}
{'_id': 'Poland', 'ReturnCount': 11}
{'_id': 'Sweden', 'ReturnCount': 11}
{'_id': 'Finland', 'ReturnCount': 10}
{'_id': 'Channel Islands', 'ReturnCount': 10}
{'_id': 'Denmark', 'ReturnCount': 9}
{'_id': 'Cyprus', 'ReturnCount': 8}
{'_id': 'Netherlands', 'ReturnCount': 8}
{'_id': 'Singapore', 'ReturnCount': 7}
{'_id': 'Czech Republic', 'ReturnCount': 5}
{'_id': 'Hong Kong', 'ReturnCount': 4}
{'_id': 'Austria', 'ReturnCount': 3}
{'_id': 'Israel', 'ReturnCount': 2

# 🔵 Performance Optimization

## Partitioning

In [ ]:
import shutil
import os

output_path = "/content/drive/MyDrive/Big Data/Assignment 02/gold_parquet"

# Remove the directory if it already exists
if os.path.exists(output_path):
    shutil.rmtree(output_path)
    print(f"Removed existing directory: {output_path}")

gold_df = df_features # Re-assign gold_df to df_features before writing

gold_df.write.partitionBy("Country") \
    .mode("overwrite") \
    .parquet(output_path)

print("✅ Gold Layer Partitioned by Country Created")

Removed existing directory: /content/drive/MyDrive/Big Data/Assignment 02/gold_parquet
✅ Gold Layer Partitioned by Country Created


## Caching

In [ ]:
gold_df.cache()
gold_df.count()

534123

## Shuffle Optimization

In [45]:
spark.conf.set("spark.sql.shuffle.partitions",50)

## MongoDB index impact analysis

In [48]:
db.gold_df.create_index([("CustomerID", 1)])
db.gold_df.create_index([("StockCode", 1)])
db.gold_df.create_index([("Country", 1)])
db.gold_df.create_index([("InvoiceDate", 1)])

'InvoiceDate_1'

In [51]:
db.fact_invoices.find({ "CustomerID": 14646 }).explain()

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'assignment2_db.fact_invoices',
  'parsedQuery': {'CustomerID': {'$eq': 14646}},
  'indexFilterSet': False,
  'queryHash': 'B6E1B21F',
  'planCacheShapeHash': 'B6E1B21F',
  'planCacheKey': '0A6B1673',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'COLLSCAN',
   'filter': {'CustomerID': {'$eq': 14646}},
   'direction': 'forward'},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned': 2081,
  'executionTimeMillis': 401,
  'totalKeysExamined': 0,
  'totalDocsExamined': 534123,
  'executionStages': {'isCached': False,
   'stage': 'COLLSCAN',
   'filter': {'CustomerID': {'$eq': 14646}},
   'nReturned': 2081,
   'executionTimeMillisEstimate': 394,
   'works': 534124,
   'advanced': 2081,
   'needTime': 532042,
   'needYield

# 🔵 Bonus Component

## Customer cohort retention analysis

In [52]:
from pyspark.sql.functions import *

cohort_df = gold_df.filter(col("Quantity") > 0)

cohort_df = cohort_df.withColumn(
    "InvoiceYearMonth",
    concat_ws("-",
              year("InvoiceDate"),
              month("InvoiceDate"))
)

first_purchase = cohort_df.groupBy("CustomerID") \
    .agg(min("InvoiceYearMonth").alias("CohortMonth"))

cohort_df = cohort_df.join(first_purchase,"CustomerID")

cohort_df = cohort_df.withColumn(
    "CohortIndex",
    (year("InvoiceDate")*12 + month("InvoiceDate")) -
    (year(to_date(concat(col("CohortMonth"),lit("-01"))))*12 +
     month(to_date(concat(col("CohortMonth"),lit("-01")))))
)

retention = cohort_df.groupBy(
    "CohortMonth","CohortIndex"
).agg(countDistinct("CustomerID").alias("Customers"))

retention.show()

+-----------+-----------+---------+
|CohortMonth|CohortIndex|Customers|
+-----------+-----------+---------+
|    2011-12|          0|      153|
|     2011-3|          6|       48|
|     2011-4|          1|       28|
|    2011-10|         -5|      171|
|    2011-10|         -8|       94|
|     2011-2|          3|       27|
|    2011-11|         -3|      118|
|    2011-10|         -1|      228|
|     2011-1|          8|      125|
|    2011-11|         -8|       96|
|    2010-12|         11|      445|
|    2011-12|         -4|       20|
|     2011-8|          0|      174|
|    2010-12|          0|     1044|
|    2010-12|          9|      350|
|     2011-3|          3|       29|
|     2011-9|          0|      249|
|     2011-2|          6|       27|
|     2011-2|          1|       25|
|    2010-12|          8|      313|
+-----------+-----------+---------+
only showing top 20 rows
